# Memory Graph Explorer

Interactive visualization of the memory graph. Hover on **nodes** to see memory statements and on **edges** to see relationship labels.

In [ ]:
import subprocess, sys, os

# Ensure dependencies are installed in the current kernel
import pyvis
import networkx

# Add src/ to path so we can import project modules
sys.path.insert(0, os.path.join(os.getcwd(), "src"))
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))

from database.repository import Repository

In [ ]:
# --- Configuration ---
# Set the username whose graph you want to explore
USERNAME = "<YOUR_USERNAME>"
DATABASE_NAME = "conversations.db"

# Database path (relative to project root)
DB_PATH = os.path.join(os.path.dirname(os.getcwd()), DATABASE_NAME)
if not os.path.exists(DB_PATH):
    # Fallback: notebook opened from project root
    DB_PATH = os.path.join(os.getcwd(), DATABASE_NAME)

print(f"Database: {DB_PATH}")
print(f"Exists:   {os.path.exists(DB_PATH)}")

In [ ]:
# Connect to the database and fetch data
repo = Repository(f"sqlite:///{DB_PATH}")
user = repo.get_user_by_username(USERNAME)

if not user:
    raise SystemExit(f"User '{USERNAME}' not found. Available users can be checked in the database.")

memories = repo.get_user_active_memories(user.id, limit=None)
mem_map = {m.id: m for m in memories}
edges = repo.get_edges_for_memories(list(mem_map.keys()))
# Keep only edges whose both endpoints are active
edges = [e for e in edges if e.source_memory_id in mem_map and e.target_memory_id in mem_map]

print(f"User:     {user.username} (id={user.id})")
print(f"Memories: {len(memories)} active")
print(f"Edges:    {len(edges)} active")

In [ ]:
from pyvis.network import Network
import networkx as nx

# Color palette per memory type
TYPE_COLORS = {
    "semantic":   "#22d3ee",  # cyan
    "episodic":   "#facc15",  # yellow
    "procedural": "#4ade80",  # green
    "affective":  "#c084fc",  # purple
}
TYPE_LABELS = {
    "semantic":   "S",
    "episodic":   "E",
    "procedural": "P",
    "affective":  "A",
}

# Build networkx graph
G = nx.Graph()

connected_ids = set()
for e in edges:
    connected_ids.add(e.source_memory_id)
    connected_ids.add(e.target_memory_id)

for nid in connected_ids:
    mem = mem_map[nid]
    G.add_node(
        nid,
        label=str(nid),
        title=(
            f"#{nid} [{mem.memory_type.upper()}]"
            f"<i>{mem.statement}</i><br><br>"
            f"Tags: {', '.join(mem.get_tags()) or 'none'}<br>"
            f"Created: {mem.created_at.strftime('%Y-%m-%d %H:%M')}"
        ),
        color=TYPE_COLORS.get(mem.memory_type, "#888888"),
        size=18,
        font={"size": 14, "color": "white", "strokeWidth": 2, "strokeColor": "#333"},
    )

for e in edges:
    G.add_edge(
        e.source_memory_id,
        e.target_memory_id,
        title=e.label,
        label=e.label,
        color="#555555",
        font={"size": 9, "color": "#aaaaaa", "strokeWidth": 0, "align": "middle"},
    )

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
isolated = [m for m in memories if m.id not in connected_ids]
if isolated:
    print(f"({len(isolated)} memories not connected by edges — not shown)")

In [ ]:
# Build interactive pyvis visualization
net = Network(
    height="700px",
    width="100%",
    bgcolor="#1a1a2e",
    font_color="white",
    notebook=True,
    cdn_resources="in_line",
)

net.from_nx(G)

# Physics settings for a nice spread layout
net.set_options("""
{
  "physics": {
    "forceAtlas2Based": {
      "gravitationalConstant": -60,
      "centralGravity": 0.008,
      "springLength": 180,
      "springConstant": 0.06,
      "damping": 0.5
    },
    "solver": "forceAtlas2Based",
    "stabilization": {
      "iterations": 200
    }
  },
  "edges": {
    "smooth": {
      "type": "continuous"
    },
    "width": 1.5
  },
  "interaction": {
    "hover": true,
    "tooltipDelay": 100,
    "zoomView": true,
    "dragNodes": true
  }
}
""")

# Render
net.show("memory_graph.html")

---

**Legend:**
| Color | Type |
|-------|------|
| 🟦 Cyan | Semantic — facts, knowledge |
| 🟨 Yellow | Episodic — events, experiences |
| 🟩 Green | Procedural — how-to, processes |
| 🟪 Purple | Affective — emotions, preferences |

**Tips:** Hover nodes for memory details. Hover edges for relationship labels. Drag nodes to rearrange. Scroll to zoom.